In [ ]:
# clone into the repo
# install requirements
# start worker (worker gets gpu status, loads weights to file, gets cloudflare url, registers itself)
# is able to take requests
!git clone https://github.com/prava241/vllm-router.git
%cd vllm-router
!pip install -r requirements.txt

# Log in to Hugging Face (meta-llama/Llama-3.1-8B-Instruct is a gated model)
!huggingface-cli login

# Download cloudflared into src/worker, since that's where worker.py runs from
%cd src/worker
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
%cd ../..

In [ ]:
# pull latest fixes from the dev branch (push your local changes to dev first)
!git pull origin dev

Before running the next cell: on your local machine (next to `server.py`), start the controller and expose it publicly so this Colab worker can reach it:

```
uvicorn src.server:app --host 0.0.0.0 --port 8000
./cloudflared tunnel --url http://localhost:8000
```

Copy the printed `https://*.trycloudflare.com` URL and paste it into `CONTROLLER_URL` below.

In [ ]:
CONTROLLER_URL = "https://REPLACE-ME.trycloudflare.com"

%cd src/worker
!python worker.py --controller-url {CONTROLLER_URL} --host 0.0.0.0 --port 8000